In [15]:
import pandas as pd
path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"


In [16]:
df_head = pd.read_csv(path, nrows=5000, low_memory=False)
df_head.head()
df_head.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 151 entries, id to settlement_term
dtypes: float64(114), int64(1), object(36)
memory usage: 5.8+ MB


In [17]:


path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"

# רשימת העמודות שאנחנו צריכים לבדוק + עמודת התאריך
columns_to_load = ['issue_d', 'desc', 'emp_title', 'title']

# הגדרת גודל "חתיכה" לקריאה (למשל 100,000 שורות בכל פעם)
chunk_size = 100000

# רשימה לאחסון התוצאות מכל חתיכה
results = []

print("Starting to process file in chunks...")

try:
    # קריאת הקובץ בחלקים
    for chunk in pd.read_csv(path, usecols=columns_to_load, chunksize=chunk_size, low_memory=False):
        
        # המרת עמודת התאריך לפורמט datetime כדי לחלץ שנה
        chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
        chunk['year'] = chunk['issue_d'].dt.year
        
        # ניקוי שורות שבהן השנה לא תקינה (אם יש כאלו)
        chunk = chunk.dropna(subset=['year'])
        chunk['year'] = chunk['year'].astype(int)
        
        # ספירה כמה ערכים אינם null לכל שנה בתוך ה-chunk הנוכחי
        summary = chunk.groupby('year')[['desc', 'emp_title', 'title']].count()
        
        # ספירה כללית של סך כל השורות באותה שנה (כדי לדעת אחוזים בהמשך)
        summary['total_rows'] = chunk.groupby('year')['issue_d'].count()
        
        results.append(summary)

    # איחוד כל התוצאות מהחלקים השונים
    final_summary = pd.concat(results).groupby(level=0).sum()

    # חישוב אחוזי מלאות (אופציונלי אך מומלץ)
    for col in ['desc', 'emp_title', 'title']:
        final_summary[f'{col}_fill_rate_%'] = (final_summary[col] / final_summary['total_rows'] * 100).round(2)

    print("\n--- סיכום מלאות עמודות לפי שנה ---")
    print(final_summary)

    # שמירת הסיכום לקובץ קטן כדי שתוכל להחליט אילו שנים להשאיר
    # final_summary.to_csv("columns_completeness_by_year.csv")

except Exception as e:
    print(f"An error occurred: {e}")

Starting to process file in chunks...


C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\581291263.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\581291263.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\581291263.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\


--- סיכום מלאות עמודות לפי שנה ---
       desc  emp_title   title  total_rows  desc_fill_rate_%  \
year                                                           
2007    588        549     603         603             97.51   
2008   2393       2296    2392        2393            100.00   
2009   5121       4986    5280        5281             96.97   
2010   8412      11793   12526       12537             67.10   
2011  12726      20285   21721       21721             58.59   
2012  32746      50195   53365       53367             61.36   
2013  48732     126249  134808      134814             36.15   
2014  15279     222393  235629      235629              6.48   
2015     45     397221  420963      421095              0.01   
2016     23     405914  411234      434407              0.01   
2017      0     411235  443579      443579              0.00   
2018      0     440583  495242      495242              0.00   

      emp_title_fill_rate_%  title_fill_rate_%  
year              

C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\581291263.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')


חיתוך לפי השנים בהן עמודה DESC קיימת יותר.

In [20]:

input_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"
output_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\filtered_by_years_only.csv"

# הגדרת השנים (כולל 2014 כדי שתוכל לבדוק את השינוי במדיניות)
target_years = [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]
chunk_size = 100000
first_chunk = True

print("Starting process: Filtering by years only (keeping all rows)...")

for chunk in pd.read_csv(input_path, chunksize=chunk_size, low_memory=False):
    # המרת תאריך וסינון שנים
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
    chunk['year'] = chunk['issue_d'].dt.year
    
    # סינון לפי שנים בלבד - בלי dropna!
    filtered_chunk = chunk[chunk['year'].isin(target_years)].copy()
    
    if not filtered_chunk.empty:
        # שמירה לקובץ החדש
        filtered_chunk.to_csv(output_path, mode='a', index=False, header=first_chunk)
        first_chunk = False

print(f"Done! Saved all rows for years {target_years} to: {output_path}")

Starting process: Filtering by years only (keeping all rows)...


C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\662350982.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\662350982.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\662350982.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')
C:\Users\ariel\AppData\Local\Temp\

Done! Saved all rows for years [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014] to: C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\filtered_by_years_only.csv


C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\662350982.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], errors='coerce')


In [22]:
import os

# וודא שהנתיב תואם לקובץ החדש שיצרת בשלב הקודם
file_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\filtered_by_years_only.csv"

if os.path.exists(file_path):
    # 1. בדיקת גודל הקובץ במגה-בייט
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

    # 2. ספירת שורות בצורה יעילה (ללא טעינה לזיכרון)
    with open(file_path, 'r', encoding='utf-8') as f:
        num_lines = sum(1 for line in f) - 1 # פחות 1 עבור שורת הכותרת

    print(f"--- תוצאות בדיקה ---")
    print(f"File Size: {file_size_mb:.2f} MB")
    print(f"Number of Rows: {num_lines:,}")
else:
    print("הקובץ לא נמצא. וודא שהרצת את קוד הסינון והנתיב נכון.")

--- תוצאות בדיקה ---
File Size: 982.14 MB
Number of Rows: 1,382,483


סינון עמודות ריקות - מעל 90 אחוז ריק + סינון עמודות שגורמות לדליפת מידע

In [23]:
input_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\filtered_by_years_only.csv"
output_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\filtered_50_cols.csv"

# 1. טעינת דגימה כדי לזהות עמודות ריקות (חוסך זמן וזיכרון)
df_sample = pd.read_csv(input_path, nrows=100000)

# זהוי עמודות שמעל 90% מהן ריקות
NA_threshold = 0.9
cols_to_drop_due_to_nas = [c for c in df_sample.columns if df_sample[c].isnull().mean() > NA_threshold]

# 2. רשימת עמודות "מידע מהעתיד" או טכניות שבוודאות לא רלוונטיות למחקר על הוגנות
# (אלו עמודות שמתעדות תשלומים, גבייה, ועיקולים שקרו אחרי מתן ההלוואה)
future_info_cols = [
    'id', 'member_id', 'url', 'funded_amnt', 'funded_amnt_inv', 
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 
    'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'policy_code', 'pymnt_plan', 'hardship_flag', 'disbursement_method'
]

# איחוד כל העמודות להסרה (בלי העמודות שרצית לשמור: desc, title, emp_title)
cols_to_keep_anyway = ['desc', 'title', 'emp_title']
total_drop = [c for c in set(cols_to_drop_due_to_nas + future_info_cols) if c not in cols_to_keep_anyway]

print(f"Dropping {len(total_drop)} irrelevant columns...")

# 3. קריאה חוזרת של הקובץ וסינון (הפעם נטען את כל השורות אבל רק את העמודות שנותרו)
all_columns = df_sample.columns
final_cols = [c for c in all_columns if c not in total_drop]

# קריאת הקובץ המלא עם העמודות הנבחרות בלבד
df_final = pd.read_csv(input_path, usecols=final_cols, low_memory=False)

# שמירה לקובץ החדש
df_final.to_csv(output_path, index=False)

print(f"Done! New file saved with {len(df_final.columns)} columns.")
print(f"Final Shape: {df_final.shape}")

C:\Users\ariel\AppData\Local\Temp\ipykernel_26436\277628874.py:5: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sample = pd.read_csv(input_path, nrows=100000)


Dropping 72 irrelevant columns...
Done! New file saved with 80 columns.
Final Shape: (1382483, 80)
